# Exploratory Data Analysis (EDA) on World Graph
This notebook loads the completed `Graph` and `People` objects from Phase 1 and explores the spatial graph and node visitation counts.

In [22]:
import sys
import os
from pathlib import Path

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from pipeline.run_logic.ast_runner import load_step_module
phase_dir = Path(project_root) / "pipeline" / "phases" / "phase_01_build_world--Lucas_Starkey"
load_step_module(phase_dir, phase_dir / "steps" / "step_01_build_devices.py")
load_step_module(phase_dir, phase_dir / "steps" / "step_03_resolve_people.py")
load_step_module(phase_dir, phase_dir / "steps" / "step_04_build_graph.py")
load_step_module(phase_dir, phase_dir / "steps" / "step_05_build_journeys.py")
load_step_module(phase_dir, phase_dir / "steps" / "step_06_interpolate_paths.py")

from pipelineio.state import load_draft

run_id = os.environ.get('PIPELINE_RUN_ID', 'EXAMPLE_RUN_ID')
graph = load_draft(f'../data/artifacts/runs/{run_id}/world/final_graph.pkl')
journeys = load_draft(f'../data/artifacts/runs/{run_id}/world/final_journeys.pkl')
people = load_draft(f'../data/artifacts/world_drafts/03_people.pkl')

print(f"Graph loaded! Nodes: {len(graph.nodes)}")
print(f"Journeys loaded! Total Trips: {len(journeys.journeys)}")


Graph loaded! Nodes: 25
Journeys loaded! Total Trips: 112755


## Top 15 Access Points by Unique Visitors
Visualizing the most actively visited access points based on the computed node_counts.

In [23]:
import plotly.express as px
import pandas as pd

if 'graph' in locals() and hasattr(graph, 'node_counts'):
    node_counts = graph.node_counts
    
    # Sort nodes by counts descending
    top_nodes = sorted(node_counts.items(), key=lambda x: x[1], reverse=True)[:15]
    
    df = pd.DataFrame(top_nodes, columns=['Access Point', 'Unique People'])
    
    # Create interactive bar chart
    fig = px.bar(
        df, 
        x='Access Point', 
        y='Unique People', 
        title='Top 15 WAPs by Unique People Count',
        color='Unique People',
        color_continuous_scale='Teal'
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()
else:
    print("Graph node_counts not available.")


## Device Metrics
Calculating the average number of devices possessed per person.

In [24]:
if 'people_obj' in locals() and hasattr(people_obj, 'people'):
    total_people = len(people_obj.people)
    total_devices = sum((person.numDevices for person in people_obj.people))
    
    avg_devices = total_devices / total_people if total_people > 0 else 0
    
    print("-" * 40)
    print("DEVICE TO PERSON CALCULATION")
    print("-" * 40)
    print(f"Total Unique People Tracked: {total_people:,}")
    print(f"Total Unique Devices Tracked: {total_devices:,}")
    print(f"Average Devices Per Person:   {avg_devices:.2f}")
    print("-" * 40)
else:
    print("People object not available.")


People object not available.


## Trace Analysis
Looking at the depth of traces for a specific highly-visited AP.

In [25]:
if 'graph' in locals() and True and top_nodes:
    top_wap = top_nodes[0][0]
    traces = [j for j in journeys.journeys if j.waypoints and j.waypoints[0].wap_id == top_wap]
    
    print(f"Total recorded visits at {top_wap}: {len(traces)}")
    print("Sample of first 5 visits:")
    for journey in traces[:5]:
        print(f" - Person Device: {journey.person_id}")
        wp = journey.waypoints[0]
        print(f"   Traced from: {wp.timestamp} ms")
else:
    print("Journeys not available.")


Total recorded visits at CLEM-Outdoor_TennisPole: 23230
Sample of first 5 visits:
 - Person Device: Client 00001
   Traced from: 1771287846000 ms
 - Person Device: Client 00001
   Traced from: 1771288143000 ms
 - Person Device: Client 00001
   Traced from: 1771525514000 ms
 - Person Device: Client 00001
   Traced from: 1771526458000 ms
 - Person Device: Client 00004
   Traced from: 1771205484000 ms


## Journey Interpolator Verification
Let's check how well the interpolator did its job by finding some journeys that have inferred (interpolated) waypoints.

In [26]:
interpolated_journeys = []
for j in journeys.journeys:
    # Check if this journey has any waypoints that were generated by the interpolator
    if any(getattr(wp, 'is_inferred', False) for wp in j.waypoints):
        interpolated_journeys.append(j)

print(f"Total journeys with interpolated (inferred) waypoints: {len(interpolated_journeys)}")
if interpolated_journeys:
    print("\nExample of an interpolated journey:")
    sample_j = interpolated_journeys[0]
    print(f"Person ID: {sample_j.person_id}")
    for wp in sample_j.waypoints:
        tag = '[INFERRED]' if getattr(wp, 'is_inferred', False) else '[OBSERVED]'
        print(f"  {wp.timestamp} - {wp.wap_id} {tag}")


Total journeys with interpolated (inferred) waypoints: 84

Example of an interpolated journey:
Person ID: Client 00013
  1771995617000 - AIEB-RM255-1 [OBSERVED]
  1771995617000 - AIEB-RM255-1 [OBSERVED]
  1771995743000 - AIEB-RM255-1 [OBSERVED]
  1771995743000 - AIEB-RM255-1 [OBSERVED]
  1771996301000.0 - BRUN-429 [INFERRED]
  1771996859000.0 - CLEM-RM216 [INFERRED]
  1771997417000 - AIEB-RM261 [OBSERVED]
  1771997417000 - AIEB-RM261 [OBSERVED]


## Unresolved Teleportations
Now let's find transitions between waypoints that are NOT physically adjacent in the graph, meaning the interpolator failed to resolve them (e.g. gap > 30 minutes, or no valid path exist).

In [ ]:
def is_adjacent(physical_edges, n1, n2):
    return n1 in physical_edges and n2 in physical_edges[n1]

teleportations = []
adj_matrix = graph.physical_edges

for j in journeys.journeys:
    for i in range(len(j.waypoints) - 1):
        wp = j.waypoints[i]
        next_wp = j.waypoints[i+1]
        
        if wp.wap_id == next_wp.wap_id:
            continue
            
        if not is_adjacent(adj_matrix, wp.wap_id, next_wp.wap_id):
            gap_mins = (next_wp.timestamp - wp.timestamp) / 1000 / 60
            teleportations.append({
                'person_id': j.person_id,
                'from_wap': wp.wap_id,
                'to_wap': next_wp.wap_id,
                'gap_mins': gap_mins,
                'from_time': wp.timestamp,
                'to_time': next_wp.timestamp
            })

print(f"Total unresolved teleportations found: {len(teleportations)}")
if teleportations:
    import pandas as pd
    tele_df = pd.DataFrame(teleportations)
    print("\nTop 10 long-gap teleportations:")
    display(tele_df.sort_values('gap_mins', ascending=False).head(10))


Total unresolved teleportations found: 0


: 